# 04 — Full RAG Pipeline

In this notebook we will:
1. Connect our retriever (ChromaDB) with an LLM (Groq)
2. Build a prompt template that feeds retrieved context to the LLM
3. Create a RAG chain using LangChain
4. Ask end-to-end questions about the earnings call
5. Compare RAG answers vs. LLM-only answers (no context)

### Key concepts
- **RAG chain**: Retrieval → Context injection → LLM generation. The LLM only sees the chunks we give it.
- **Prompt template**: Instructions + retrieved context + user question. How you structure this controls the quality of answers.
- **Grounding**: When the LLM answers based on the provided context, not its training data. This is what makes RAG useful.
- **Hallucination**: When the LLM invents information not in the context. A well-designed prompt reduces this.

## Setup

In [1]:
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv("../.env")
print("Environment loaded")

Environment loaded


In [2]:
# Connect to ChromaDB
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
client = chromadb.PersistentClient(path="../chroma_db")
collection = client.get_collection(
    name="earnings_calls",
    embedding_function=embedding_fn,
)
print(f"Collection: {collection.name} ({collection.count()} documents)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Collection: earnings_calls (75 documents)


In [ ]:
# Connect to Groq LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.5,
)

# Quick test
response = llm.invoke("Say hello in one sentence.")
print(f"LLM connected: {response.content}")

LLM connected: Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


## Step 1: Build the retriever function

A helper that takes a query and returns formatted context from ChromaDB.

In [4]:
def retrieve(query: str, n_results: int = 5, where: dict = None) -> str:
    """Retrieve relevant chunks and format them as context for the LLM."""
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        where=where,
    )
    
    context_parts = []
    for i in range(len(results["documents"][0])):
        doc = results["documents"][0][i]
        meta = results["metadatas"][0][i]
        speaker = meta.get("speaker", "Unknown")
        role = meta.get("role", "Unknown")
        section = meta.get("section", "Unknown")
        context_parts.append(
            f"[{speaker} — {role} | {section}]\n{doc}"
        )
    
    return "\n\n---\n\n".join(context_parts)

# Test it
ctx = retrieve("What were the revenue results?")
print(ctx[:500] + "...")

[Kevan Parekh — Senior Vice President, Chief Financial Officer | prepared_remarks]
Thanks, Tim, and good afternoon, everyone. I'm going to cover the results for the first quarter of our fiscal year. We are very pleased to report an all-time high for revenue, with December quarter revenue of $124.3 billion, up 4% year over year. We achieved all-time revenue records in the Americas, Europe, Japan, and rest of Asia Pacific and grew in the vast majority of markets we track.
Products revenue was $98 ...


## Step 2: Build the prompt template

This is the core of the RAG pipeline — the instructions that tell the LLM how to use the context.

---

Write the system prompt for the RAG chain. The template receives two variables: `{context}` (the retrieved chunks) and `{question}` (the user's question).

Think about:
- What should the LLM do with the context?
- What should it do when the context doesn't contain the answer?
- Should it cite who said what?

In [17]:
RAG_PROMPT_TEMPLATE = """
    ### Role
    You are a Senior Forensic Financial Analyst. Your goal is to provide a precise, objective, and data-driven analysis of company performance based ONLY on the provided earning call transcripts and financial data.

    ### Task
    Analyze the provided context to answer the user's question. Focus on:
    1. **Financial Results:** Revenue, EPS, and specific segment performance.
    2. **Management Guidance:** Future projections and strategic plans.
    3. **Sentiment & Nuance:** Distinguish between confident prepared remarks and potentially "hedged" or vague answers during the Q&A section.

    ### Context
    {context}

    ### Constraints (Critical)
    - **Groundedness:** If the information is not present in the {context}, state clearly: "I cannot find specific data regarding [X] in the provided transcripts." Do not invent numbers.
    - **Recency Note:** If the provided data does not cover the current month/quarter, explicitly inform the user that the answer is based on the most recent available transcript (specify the date/quarter if possible).
    - **No Fluff:** Avoid marketing language. If a company describes results as "robust," verify if the numbers actually support that claim.

    ### Output Format
    Your response must follow this structure:

    1. **Executive Summary:** A 2-3 sentence direct answer to the user's question.
    2. **Detailed Analysis:**
    - **Key Metrics:** Highlight specific numbers (Revenue, Margins, etc.).
    - **Strategic Outlook:** What management promised for the next period.
    - **Analyst Perspective:** Mention any "red flags" or dodged questions observed in the Q&A.
    3. **Evidence & References:** Provide direct quotes or specific sections from the text used to generate the answer.

    ### User Question
    {question}
"""

prompt = ChatPromptTemplate.from_template(RAG_PROMPT_TEMPLATE)
print("Prompt variables:", prompt.input_variables)

Prompt variables: ['context', 'question']


## Step 3: Build the RAG chain

Now we wire everything together: retrieval → prompt → LLM → output.

In [18]:
# The chain: prompt → LLM → parse output as string
chain = prompt | llm | StrOutputParser()

def ask(question: str, n_results: int = 5, where: dict = None) -> str:
    """Full RAG pipeline: retrieve context, then generate answer."""
    context = retrieve(question, n_results=n_results, where=where)
    answer = chain.invoke({"context": context, "question": question})
    return answer

print("RAG chain ready.")

RAG chain ready.


## Step 4: Ask questions!

Let's test the full pipeline with different types of questions.

In [19]:
# Question 1: Factual question about results
q = "What was Apple's total revenue for the quarter and how did it compare to last year?"
print(f"Q: {q}\n")
print(ask(q))

Q: What was Apple's total revenue for the quarter and how did it compare to last year?

1. **Executive Summary:** Apple reported a total revenue of $124.3 billion for the December quarter, which represents a 4% increase from the same period last year. This achievement marks an all-time record for the company. The revenue growth was driven by strong performances across various segments, including services and products.

2. **Detailed Analysis:**
    - **Key Metrics:** The total revenue was $124.3 billion, with a 4% year-over-year increase. Services revenue reached an all-time record of $26.3 billion, up 14% year over year, and products revenue was $98 billion, up 2% year over year.
    - **Strategic Outlook:** Management expects the March quarter total company revenue to grow low to mid single digits year over year, despite a headwind from foreign exchange. Services revenue is expected to grow low double digits year over year.
    - **Analyst Perspective:** Analysts questioned the susta

In [14]:
# Question 2: Question about a specific person
q = "What did the CFO say about gross margins?"
print(f"Q: {q}\n")
print(ask(q, where={"role": "Senior Vice President, Chief Financial Officer"}))

Q: What did the CFO say about gross margins?

1. **Overall Summary**: The CFO, Kevan Parekh, discussed the company's gross margins, stating that they are guiding to a range of 46.5% to 47.5% for the next quarter. He attributed the strong sequential improvement in gross margins to favorable mix and leverage, as well as a favorable commodity environment. The company's services business also had a strong performance, with a gross margin of 75% in the December quarter.

2. **More Details**: The CFO mentioned that the company's product launches and the mix of products sold have a significant impact on gross margins. He noted that customers are gravitating towards the company's pro products, which have favorable gross margins. The company also benefits from a favorable commodity environment, which helps to reduce costs. However, the CFO warned that foreign exchange headwinds will have a negative impact on gross margins in the next quarter. Despite this, the company is confident in its guidan

In [15]:
# Question 3: Question about a specific topic across speakers
q = "What was discussed about Apple Intelligence and its impact?"
print(f"Q: {q}\n")
print(ask(q))

Q: What was discussed about Apple Intelligence and its impact?

1. **Overall Summary**: The discussion about Apple Intelligence centered around its impact on iPhone demand and its features. According to the earnings call, Apple Intelligence has contributed to stronger year-over-year performance of the iPhone 16 family in markets where it is available. The features of Apple Intelligence, such as Writing Tools, Image Playground, Genmoji, visual intelligence, and Clean Up, are being widely used by consumers.

2. **More Details**: The earnings call transcripts reveal that Apple Intelligence has played a significant role in driving iPhone sales, particularly in markets where it has been launched. The company's CEO, Tim Cook, mentioned that the year-over-year performance of the iPhone 16 family was stronger in markets where Apple Intelligence was available. Additionally, the features of Apple Intelligence, such as Writing Tools and Image Playground, are popular among consumers. The company a

## Step 5: RAG vs. No Context

Let's see what happens when we ask the same question to the LLM *without* any context. This shows why RAG matters.

In [16]:
q = "What was Apple's gross margin in Q1 2025 and what drove it?"

# With RAG
print("WITH RAG (grounded in transcript):")
print("=" * 50)
print(ask(q))

print("\n")

# Without RAG — just the LLM
print("WITHOUT RAG (LLM only, no context):")
print("=" * 50)
no_context_answer = llm.invoke(
    f"Answer this question about Apple's Q1 2025 earnings call: {q}"
)
print(no_context_answer.content)

print("\n")
print("Notice: The RAG answer cites specific numbers from the transcript.")
print("The LLM-only answer might be vague, outdated, or hallucinated.")

WITH RAG (grounded in transcript):
1. **Overall Summary**: Unfortunately, the provided transcript does not explicitly mention Apple's gross margin for Q1 2025. However, based on the information given, Apple has been focusing on innovation, including the rollout of Apple Intelligence, which has shown positive indicators in markets where it has been introduced. The company has also seen record revenue in various segments, including services and iPhone sales.

2. **More Details**: During the earnings call, Timothy Donald Cook mentioned that markets where Apple Intelligence was rolled out performed better on a year-over-year basis than those where it was not. This suggests that Apple Intelligence could be a driver for future growth. Additionally, the company reported an all-time record revenue of $124.3 billion for the December quarter, with EPS also setting an all-time record of $2.40, 10% higher year over year. The services segment achieved an all-time revenue record of $26.3 billion, gr

## Summary

In this notebook you learned:
- How to connect a retriever (ChromaDB) to an LLM (Groq) through a prompt template
- The prompt template is where you control grounding, citations, and hallucination prevention
- Metadata filters let you scope questions to specific speakers or sections
- RAG provides grounded, verifiable answers — unlike the LLM alone